<a href="https://colab.research.google.com/github/luizalmin-netizen/loja/blob/main/loja_da_sih.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mercadopago paypalrestsdk Flask

In [3]:
# Install pyngrok to expose the Flask app
!pip install pyngrok

### Running the Flask App with ngrok

Google Colab runs your code on a remote virtual machine, which is not directly accessible from your local browser. To make your Flask application accessible, we can use `ngrok` to create a secure tunnel from a public URL to your local server running in Colab.

Here's how it works:
1. **`pyngrok` installation**: We've just installed the `pyngrok` library.
2. **Import `ngrok`**: We'll import `ngrok` into your application.
3. **Start the tunnel**: Inside your Flask app's `if __name__ == '__main__':` block, we'll start an `ngrok` tunnel on the port your Flask app is running (default is 5000).
4. **Public URL**: `ngrok` will provide a public URL that you can open in your browser to interact with your Flask application.

I will now modify your main application cell (`Qm6T6nKfQIP4`) to include the `ngrok` setup.

In [4]:
from flask import Flask, render_template, request, redirect, url_for, session
import sqlite3
import mercadopago
import paypalrestsdk
from pyngrok import ngrok
import os

app = Flask(__name__)
app.secret_key = "segredo_simone_modas"

# ---------------------------
# Banco de dados SQLite
# ---------------------------
def get_produtos():
    conn = sqlite3.connect("loja.db")
    conn.row_factory = sqlite3.Row
    produtos = conn.execute("SELECT * FROM produtos").fetchall()
    conn.close()
    return produtos

# ---------------------------
# Rotas principais
# ---------------------------
@app.route("/")
def home():
    produtos = get_produtos()
    return render_template("index.html", produtos=produtos)

@app.route("/adicionar/<int:produto_id>")
def adicionar(produto_id):
    carrinho = session.get("carrinho", {})
    carrinho[str(produto_id)] = carrinho.get(str(produto_id), 0) + 1
    session["carrinho"] = carrinho
    return redirect(url_for("ver_carrinho"))

@app.route("/remover/<int:produto_id>")
def remover(produto_id):
    carrinho = session.get("carrinho", {})
    if str(produto_id) in carrinho:
        del carrinho[str(produto_id)]
    session["carrinho"] = carrinho
    return redirect(url_for("ver_carrinho"))

@app.route("/carrinho")
def ver_carrinho():
    carrinho = session.get("carrinho", {})
    itens = []
    total = 0
    produtos = get_produtos()
    for pid, qtd in carrinho.items():
        produto = next((p for p in produtos if p["id"] == int(pid)), None)
        if produto:
            subtotal = produto["preco"] * qtd
            itens.append({
                "id": produto["id"],
                "nome": produto["nome"],
                "quantidade": qtd,
                "preco": produto["preco"],
                "subtotal": subtotal,
                "imagem": produto["imagem"]
            })
            total += subtotal
    return render_template("carrinho.html", itens=itens, total=total)

@app.route("/checkout", methods=["GET", "POST"])
def checkout():
    if request.method == "POST":
        nome = request.form.get("nome")
        email = request.form.get("email")
        codigo = request.form.get("codigo")

        # calcula total do carrinho
        carrinho = session.get("carrinho", {})
        produtos = get_produtos()
        total = 0
        for pid, qtd in carrinho.items():
            produto = next((p for p in produtos if p["id"] == int(pid)), None)
            if produto:
                total += produto["preco"] * qtd

        # aplica desconto
        if codigo == "VINTAGE10":
            total *= 0.9

        session["carrinho"] = {}  # limpa carrinho
        return f"<h1>Obrigado pela compra, {nome}!</h1><p>Total com desconto: R$ {total}</p><p>Recibo enviado para {email}.</p>"
    return render_template("checkout.html")

# ---------------------------
# Integração Mercado Pago
# ---------------------------
sdk = mercadopago.SDK("SUA_CHAVE_DE_ACESSO")

@app.route("/pagamento")
def pagamento():
    preference_data = {
        "items": [
            {
                "title": "Compra Simone Modas",
                "quantity": 1,
                "currency_id": "BRL",
                "unit_price": 100.00  # aqui você coloca o total do carrinho
            }
        ],
        "back_urls": {
            "success": "http://localhost:5000/sucesso",
            "failure": "http://localhost:5000/falha",
            "pending": "http://localhost:5000/pendente"
        },
        "auto_return": "approved",
    }
    preference_response = sdk.preference().create(preference_data)
    preference = preference_response["response"]
    return redirect(preference["init_point"])

@app.route("/sucesso")
def sucesso():
    return "<h1>Pagamento aprovado! Obrigado pela compra 🎉</h1>"

@app.route("/falha")
def falha():
    return "<h1>Pagamento não aprovado. Tente novamente.</h1>"

@app.route("/pendente")
def pendente():
    return "<h1>Pagamento pendente. Aguarde confirmação.</h1>"

# ---------------------------
# Integração PayPal
# ---------------------------
paypalrestsdk.configure(
    {
        "mode": "sandbox",  # use "live" em produção
        "client_id": "SEU_CLIENT_ID",
        "client_secret": "SEU_CLIENT_SECRET",
    }
)

@app.route("/pagamento_paypal")
def pagamento_paypal():
    pagamento = paypalrestsdk.Payment(
        {
            "intent": "sale",
            "payer": {"payment_method": "paypal"},
            "redirect_urls": {
                "return_url": "http://localhost:5000/sucesso",
                "cancel_url": "http://localhost:5000/falha",
            },
            "transactions": [
                {
                    "item_list": {
                        "items": [
                            {
                                "name": "Compra Simone Modas",
                                "sku": "001",
                                "price": "100.00",
                                "currency": "BRL",
                                "quantity": 1,
                            }
                        ]
                    },
                    "amount": {"total": "100.00", "currency": "BRL"},
                    "description": "Pagamento de produtos Simone Modas",
                }
            ],
        }
    )

    if pagamento.create():
        for link in pagamento.links:
            if link.rel == "approval_url":
                return redirect(link.href)
    else:
        return "Erro ao criar pagamento"

# ---------------------------
# Inicialização
# ---------------------------
if __name__ == "__main__":
    # Replace "YOUR_NGROK_AUTHTOKEN" with your actual ngrok authentication token
    NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN" # You'll need to get this from ngrok.com
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Open a ngrok tunnel to the Flask app's port (5000)
    public_url = ngrok.connect(5000)
    print(f" * ngrok tunnel available at: {public_url}")

    app.run(debug=True, use_reloader=False)


ERROR:pyngrok.process.ngrok:t=2026-06-05T01:00:27+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-05T01:00:27+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-05T01:00:27+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified is an ngrok v1 authtoke

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("jessicali9530/animal-crossing-new-horizons-nookplaza-dataset")

In [12]:
# @title 📊 Animal Crossing: Logística e Revenda {display-mode: "form"}

import pandas as pd
import json
import os
import glob
import kagglehub
from IPython.display import HTML
from google.colab import output

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

def load_animal_crossing_data():
    try:
        dataset_path = kagglehub.dataset_download("jessicali9530/animal-crossing-new-horizons-nookplaza-dataset")
        all_files = glob.glob(os.path.join(dataset_path, "*.csv"))
        if not all_files: return None

        df = pd.read_csv(all_files[0])

        # Garantir colunas numéricas de forma segura
        df['Buy'] = pd.to_numeric(df['Buy'], errors='coerce').fillna(0)

        if 'Sell' in df.columns:
            df['Sell'] = pd.to_numeric(df['Sell'], errors='coerce').fillna(0)
        else:
            df['Sell'] = 0

        def calc_shipping(row):
            # Lógica de frete: 500 para itens grandes (Bridge) ou 5% do valor
            cat = str(row.get('Category', ''))
            base = 500 if 'Bridge' in cat else 50
            return int(base + (row['Buy'] * 0.05))

        df['Shipping'] = df.apply(calc_shipping, axis=1)
        df['Total_Cost'] = df['Buy'] + df['Shipping']
        df['Version'] = df['Version'].astype(str) if 'Version' in df.columns else "1.0.0"
        df = df.fillna("N/A")
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None

data_df = load_animal_crossing_data()

if data_df is not None:
    json_data = data_df.to_json(orient='records')
    html_template = """
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
        <style>
            body { font-family: 'Segoe UI', sans-serif; background-color: #f4f6f8; padding: 20px; color: #2c3e50; }
            .controls { background: white; padding: 15px; border-radius: 12px; margin-bottom: 20px; display: flex; gap: 15px; }
            .dashboard-container { display: grid; grid-template-columns: repeat(4, 1fr); grid-gap: 15px; }
            .card { background: white; padding: 15px; border-radius: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
            .kpi-value { font-size: 1.4rem; font-weight: bold; color: #1a73e8; }
            .chart-container { grid-column: span 2; height: 300px; }
            .wide-container { grid-column: span 4; }
            .scrollable { max-height: 300px; overflow-y: auto; }
            table { width: 100%; border-collapse: collapse; font-size: 0.8rem; }
            th { background: #eee; position: sticky; top: 0; padding: 8px; text-align: left; }
            td { padding: 8px; border-bottom: 1px solid #eee; }
            .badge { background: #fff3e0; color: #e65100; padding: 2px 4px; border-radius: 4px; font-weight: bold; }
            input { padding: 8px; border-radius: 6px; border: 1px solid #ddd; width: 250px; }
        </style>
    </head>
    <body>
        <div class="controls">
            <input type="text" id="search" placeholder="Buscar por nome do item...">
        </div>
        <div class="dashboard-container">
            <div class="card"><div>Total de Itens</div><div id="k1" class="kpi-value">0</div></div>
            <div class="card"><div>Média Frete Estimado</div><div id="k2" class="kpi-value">0</div></div>
            <div class="card chart-container"><canvas id="c1"></canvas></div>
            <div class="card wide-container">
                <div class="scrollable">
                    <table>
                        <thead><tr><th>Nome</th><th>Preço Compra</th><th>Frete</th><th>Custo Total</th><th>Venda</th></tr></thead>
                        <tbody id="tb"></tbody>
                    </table>
                </div>
            </div>
        </div>
        <script>
            const data = DATA_STR;
            function update(){
                const term = document.getElementById('search').value.toLowerCase();
                const filtered = data.filter(d => (d.Name || '').toLowerCase().includes(term));

                document.getElementById('k1').innerText = filtered.length.toLocaleString();
                const ship = filtered.reduce((a,b)=>a+(b.Shipping||0),0)/filtered.length || 0;
                document.getElementById('k2').innerText = Math.round(ship).toLocaleString();

                const tb = document.getElementById('tb'); tb.innerHTML = '';
                filtered.slice(0,100).forEach(d => {
                    tb.innerHTML += `<tr>
                        <td><strong>${d.Name}</strong></td>
                        <td>${(d.Buy||0).toLocaleString()}</td>
                        <td><span class="badge">+${(d.Shipping||0).toLocaleString()}</span></td>
                        <td>${(d.Total_Cost||0).toLocaleString()}</td>
                        <td>${(d.Sell||0).toLocaleString()}</td>
                    </tr>`;
                });
            }
            document.getElementById('search').oninput = update;
            update();
        </script>
    </body>
    </html>
    """.replace('DATA_STR', json_data)
    display(HTML(html_template))


Using Colab cache for faster access to the 'animal-crossing-new-horizons-nookplaza-dataset' dataset.


Nome,Preço Compra,Frete,Custo Total,Venda
